In [10]:
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [26]:
# Выводим список доступных физических устройств типа 'GPU'
gpus = tf.config.list_physical_devices('GPU')
gpus

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [27]:
if gpus:
    try:
        # Устанавливаем, чтобы TensorFlow использовал только первое GPU
        # и не выделял всю память сразу
        tf.config.experimental.set_memory_growth(gpus[0], False)
        print(f"✅ Найдена и настроена видеокарта: {gpus[0].name}")
    except RuntimeError as e:
        print(e)
else:
    print("❌ Видеокарта не найдена. TensorFlow будет использовать CPU.")


Physical devices cannot be modified after being initialized


In [5]:
# --- Основні параметри ---
DATA_DIR = 'archive/train-scene classification/train' # <-- Папка, де лежать ВСІ зображення (train + test)
TRAIN_CSV_PATH = 'archive/train-scene classification/train.csv'
TEST_CSV_PATH = 'archive/test_WyRytb0.csv'
IMG_WIDTH = 128
IMG_HEIGHT = 128
BATCH_SIZE = 32


In [6]:
# --- Завантажуємо дані з CSV ---
train_df = pd.read_csv(TRAIN_CSV_PATH)
test_df = pd.read_csv(TEST_CSV_PATH)

In [7]:
# Подивимось на дані
print("Тренувальні дані:")
train_df.head()

Тренувальні дані:


,image_name,label
0,0.jpg,0
1,1.jpg,4
2,2.jpg,5
3,4.jpg,0
4,7.jpg,4


In [8]:
print("\nТестові дані:")
test_df.head()


Тестові дані:


,image_name
0,3.jpg
1,5.jpg
2,6.jpg
3,11.jpg
4,14.jpg


In [11]:
# Створюємо повні шляхи до файлів
train_df['image_path'] = train_df['image_name'].apply(lambda x: os.path.join(DATA_DIR, x))
test_df['image_path'] = test_df['image_name'].apply(lambda x: os.path.join(DATA_DIR, x))

In [12]:
# Конвертуємо мітки в рядки (для категоріального режиму)
train_df['label'] = train_df['label'].astype(str)

In [13]:
print("\nОновлений тренувальний DataFrame:")
train_df.head()


Оновлений тренувальний DataFrame:


,image_name,label,image_path
0,0.jpg,0,archive/train-scene classification/train\0.jpg
1,1.jpg,4,archive/train-scene classification/train\1.jpg
2,2.jpg,5,archive/train-scene classification/train\2.jpg
3,4.jpg,0,archive/train-scene classification/train\4.jpg
4,7.jpg,4,archive/train-scene classification/train\7.jpg


In [14]:
# Створюємо генератор для тренувальних даних з аугментацією
train_datagen = ImageDataGenerator(
    rescale=1./255,          # Нормалізація пікселів
    validation_split=0.2,    # Виділяємо 20% даних на валідацію
    rotation_range=20,       # Випадкові повороти
    width_shift_range=0.2,   # Випадкові зміщення по ширині
    height_shift_range=0.2,  # і висоті
    horizontal_flip=True     # Випадкові горизонтальні відображення
)

In [15]:
# Створюємо генератор для тестових даних (без аугментації, тільки нормалізація)
test_datagen = ImageDataGenerator(rescale=1./255)

In [16]:
# Тренувальний потік
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='image_path', # Стовпець зі шляхами до зображень
    y_col='label',      # Стовпець з мітками
    subset='training',  # Вказуємо, що це тренувальна частина
    batch_size=BATCH_SIZE,
    seed=42,
    shuffle=True,
    class_mode='categorical', # Мітки будуть у one-hot форматі
    target_size=(IMG_HEIGHT, IMG_WIDTH)
)

Found 13628 validated image filenames belonging to 6 classes.


In [17]:
# Валідаційний потік
validation_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df, # Використовуємо той самий DataFrame
    x_col='image_path',
    y_col='label',
    subset='validation', # Вказуємо, що це валідаційна частина
    batch_size=BATCH_SIZE,
    seed=42,
    shuffle=True,
    class_mode='categorical',
    target_size=(IMG_HEIGHT, IMG_WIDTH)
)

Found 3406 validated image filenames belonging to 6 classes.


In [18]:
# Тестовий потік
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='image_path',
    y_col=None,
    batch_size=1, # Зазвичай для тестування batch_size=1
    shuffle=False, # Не перемішуємо, щоб зберегти порядок
    class_mode=None,
    target_size=(IMG_HEIGHT, IMG_WIDTH)
)

Found 7301 validated image filenames.


In [19]:
# Отримуємо кількість класів з генератора
num_classes = len(train_generator.class_indices)
num_classes

6

In [20]:
model = models.Sequential([
    # Вхідний шар
    Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    layers.MaxPooling2D(),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    # Вихідний шар з активацією softmax
    layers.Dense(num_classes, activation='softmax')
])

In [21]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 max_pooling2d (MaxPooling2D  (None, 64, 64, 3)        0         
 )                                                               
                                                                 
 conv2d (Conv2D)             (None, 64, 64, 32)        896       
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 32, 32, 32)       0         
 2D)                                                             
                                                                 
 conv2d_1 (Conv2D)           (None, 32, 32, 64)        18496     
                                                                 
 max_pooling2d_2 (MaxPooling  (None, 16, 16, 64)       0         
 2D)                                                             
                                                        

In [22]:
# --- компіляція ---
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [23]:
# --- Навчання з генераторами ---
EPOCHS = 10

history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS,
    # Розраховуємо кроки на епоху
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    validation_steps=validation_generator.samples // BATCH_SIZE
)

Epoch 1/10
165/425 [==========>...................] - ETA: 1:53 - loss: 1.2474 - accuracy: 0.5100

KeyboardInterrupt: 